# 学習なしで好きなラベルの固有表現を抽出する (GLiNER)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/20_ner/gliner.ipynb)

NLP クックブック 20-04 ／ 固有表現抽出

「病名」「薬品名」「役職」のような独自のラベルで固有表現抽出をしたいが、学習データがない。

In [1]:
!pip install -q gliner fugashi unidic-lite

In [2]:
from gliner import GLiNER
from huggingface_hub import hf_hub_download
from sentencepiece import sentencepiece_model_pb2
from tokenizers import Regex, normalizers

model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")
# 中身は mDeBERTa。transformers 5 系の DebertaV2Tokenizer は文字の正規化 (全角→半角など) を組み込まずに作られ、
# 全角の記号や英数字を [UNK] にしてしまう (issue #49622)。GLiNER の中のトークナイザーに、元の spm.model にある正規化の規則を戻す
proto = sentencepiece_model_pb2.ModelProto()
proto.ParseFromString(open(hf_hub_download(model.config.model_name, "spm.model"), "rb").read())
model.data_processor.transformer_tokenizer.backend_tokenizer.normalizer = normalizers.Sequence([
    normalizers.Strip(), normalizers.Precompiled(proto.normalizer_spec.precompiled_charsmap), normalizers.Replace(Regex(" {2,}"), " ")])

text = "Dr. Sarah Chen, CTO of Anthropic, presented the Claude model at NeurIPS 2024 in Vancouver."
for e in model.predict_entities(text, ["person", "job title", "company", "product", "conference", "city"], threshold=0.4):
    print(f"{e['text']:14}{e['label']:12}{e['score']:.2f}")

Dr. Sarah Chenperson      0.96
CTO           job title   0.73
Anthropic     company     0.96
Claude model  product     0.96
NeurIPS 2024  conference  0.98
Vancouver     city        0.98


In [3]:
# 日本語は空白で区切られていないので、分かち書きしてから渡す
from fugashi import Tagger

wakati = Tagger("-Owakati")
text = "田中医師は、インフルエンザと診断された患者にタミフルを5日分処方し、慶應義塾大学病院への紹介状を書いた。"
labels = ["人名", "病名", "薬品名", "医療機関", "期間"]
for e in model.predict_entities(wakati.parse(text).strip(), labels, threshold=0.5):
    print(f"{e['text'].replace(' ', ''):\u3000<10}{e['label']:\u3000<6}{e['score']:.2f}")

田中医師　　　　　　人名　　　　0.95
インフルエンザ　　　病名　　　　0.97
タミフル　　　　　　薬品名　　　0.70
5日分　　　　　　　期間　　　　0.83
慶應義塾大学病院　　医療機関　　0.94


- ラベル名を自然言語で与えるだけで抽出できる小型のエンコーダモデル。CPU でも動く。
- 内部で空白区切りの単語を単位にしているため、日本語や中国語は分かち書きが必要（そのまま渡すと文全体が 1 語扱いになる）。
- ラベル名の言い回しで結果が変わる。`threshold` を下げると再現率が上がる。
- LLM に抽出させる方法と比べて、高速で、原文中の位置 (`start`/`end`) が確実に得られるのが利点。
- GLiNER はトークナイザーを内部で読み込み、元のモデル (`microsoft/mdeberta-v3-base`) には `tokenizer.json` もないので、`tokenizer.json` から読み直す方法（Transformers の基本操作の章の「2 つの文の含意関係を判定する (NLI)」の注意書きを参照）が使えない。そこで、読み込んだあとのトークナイザーの正規化の部分だけを差し替えている。上流 (transformers) で直れば不要になる。